# Inspecting the noise diode excess

Example notebook for the output of `NoiseDiodeExcessPlugin`
(`museek/plugin/noise_diode_excess_plugin.py`).

The plugin measures the noise diode (ND) excess signal `V_on - V_off` from the
calibrator **track** data. Relative to the older `NoiseDiodePlugin` it:

1. **Leaks the flagging onto the ND dumps** — every flag layer except
   `noise_diode_on` and `aoflagger_tracking` is applied to the firing dumps, so a
   firing (or an adjacent off dump used for the baseline) that is contaminated
   by real RFI / antenna / elevation issues is excluded.
2. **Drops bad firings** — a firing whose on-dump survives in fewer than
   `nd_dump_good_fraction` of channels is fully masked. A physicality check also
   masks non-positive excess values (a dead diode, or a contaminated
   off-baseline driving the excess negative); if more than
   `nd_excess_failure_fraction` of a firing's channels go non-positive the whole
   firing is masked as a diode failure.
3. **Produces a robust averaged measure** — the masked median over all firings,
   per frequency and receiver (`NOISE_DIODE_EXCESS_AVERAGE`).

This notebook loads the plugin's context pickle and shows the per-firing excess,
the averaged excess spectrum, and the effect of the leaked flagging.

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma

from museek.enums.result_enum import ResultEnum

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

In [ ]:
# Update this path to your results directory.
# The plugin writes 'noise_diode_excess_plugin.pickle' to OUTPUT_PATH when do_store_context=True.
pickle_path = '/idia/users/geoffmurphy/aoflagger_param_test/1675632179/noise_diode_excess_plugin.pickle'

with open(pickle_path, 'rb') as f:
    context = pickle.load(f)

print(f'Loaded context with keys: {list(context.keys())}')

In [ ]:
track_data = context.get(ResultEnum.TRACK_DATA).result
nd_excess = context.get(ResultEnum.NOISE_DIODE_EXCESS).result            # (n_firings, n_freq, n_receivers)
nd_excess_average = context.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result  # (n_freq, n_receivers)
noise_on_timestamp = context.get(ResultEnum.NOISE_ON_TIMESTAMP).result   # (n_firings,) seconds since start

freq_MHz = track_data.frequencies.squeeze / 1e6
receiver_names = [str(r) for r in track_data.receivers]

# Limit the analysis to the first 10 receivers so this review notebook's outputs
# stay compact. Raise/remove n_receivers_show to inspect the full array.
n_receivers_show = 10
nd_excess = nd_excess[:, :, :n_receivers_show]
nd_excess_average = nd_excess_average[:, :n_receivers_show]
receiver_names = receiver_names[:n_receivers_show]

n_firings, n_freq, n_receivers = nd_excess.shape
print(f'{n_firings} noise diode firings, {n_freq} channels, {n_receivers} receivers (capped)')
print(f'Frequency range: {freq_MHz.min():.1f} - {freq_MHz.max():.1f} MHz (UHF)')
print(f'Track duration covered by firings: {noise_on_timestamp.max() / 60:.1f} min')
print(f'Overall masked fraction of the excess: {ma.getmaskarray(nd_excess).mean() * 100:.1f}%')

## Averaged noise diode excess spectrum

`NOISE_DIODE_EXCESS_AVERAGE` is the masked median over all firings — the "good
measure" intended for downstream calibration. Below it is shown for a few
receivers.

In [ ]:
# Display all (capped) receivers in the averaged spectrum.
show_receivers = list(range(n_receivers))

plt.figure()
for i_rec in show_receivers:
    plt.plot(freq_MHz, nd_excess_average[:, i_rec], lw=1, label=receiver_names[i_rec])
plt.xlabel('Frequency [MHz]')
plt.ylabel('Median ND excess  $V_{on} - V_{off}$')
plt.title('Time-averaged noise diode excess spectrum')
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.3)
plt.show()

## Per-firing excess over time (stability check)

The ND should be a stable reference. Here the per-firing excess at a clean
channel near 750 MHz is plotted against firing time, with its percent deviation
about the median.

In [ ]:
i_750 = int(np.argmin(np.abs(freq_MHz - 750.0)))

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 7))
for i_rec in show_receivers:
    series = nd_excess[:, i_750, i_rec]
    ax1.plot(noise_on_timestamp / 60, series, '.-', ms=4, lw=0.8, label=receiver_names[i_rec])
    pct = 100.0 * (series - ma.median(series)) / ma.median(series)
    ax2.plot(noise_on_timestamp / 60, pct, '.-', ms=4, lw=0.8)

ax1.set_ylabel(f'ND excess @ {freq_MHz[i_750]:.0f} MHz')
ax1.set_title('Per-firing noise diode excess')
ax1.legend(ncol=2, fontsize=8)
ax1.grid(alpha=0.3)
ax2.set_xlabel('Firing time [min since track start]')
ax2.set_ylabel('Percent deviation\nabout median [%]')
ax2.grid(alpha=0.3)
#ax1.set_xlim(140,170)
#ax2.set_xlim(140,170)
plt.tight_layout()
plt.show()

## Effect of the leaked flagging

Because the RFI / quality flags are now leaked onto the ND dumps, contaminated
firings and channels are masked out of the excess. Below: the masked fraction
per firing (averaged over receivers), and the receivers the plugin flagged for
bad noise diode behaviour.

In [ ]:
masked_fraction_per_firing = ma.getmaskarray(nd_excess).mean(axis=(1, 2))

plt.figure()
plt.plot(noise_on_timestamp / 60, masked_fraction_per_firing * 100, '.-', ms=4, lw=0.8)
plt.xlabel('Firing time [min since track start]')
plt.ylabel('Masked fraction [%]')
plt.title('Fraction of the excess masked per firing (leaked flagging)')
#plt.xlim(-1,12.5)
plt.grid(alpha=0.3)
plt.show()

# Receivers fully flagged by the plugin's 'noise_diode_bad_behavior' layer.
if 'noise_diode_bad_behavior' in track_data.flags.flag_names:
    idx = track_data.flags.flag_names.index('noise_diode_bad_behavior')
    bad = track_data.flags._flags[idx].get_array().any(axis=(0, 1))
    flagged = [receiver_names[i] for i in range(n_receivers) if bad[i]]
    print(f'Receivers flagged for bad noise diode behaviour: {flagged if flagged else "none"}')



## Per-firing excess deviation for the first period, split by pointing

The channel-median excess per firing, expressed as percent deviation about the
per-receiver mean over the first calibrator period. Firings are tagged by
pointing so drifts within the period are visible. This reuses the plugin's
already-computed (and flag-leaked) per-firing excess, so nothing is recomputed
from the raw visibility.

In [ ]:
# Map each firing to a calibrator period. The plugin runs over all of TRACK_DATA,
# so the firings axis concatenates every period (and every pointing within them)
# with no period tag. We recover the first period and split it into pointings.
periods = context.get(ResultEnum.CALIBRATOR_VALIDATED_PERIODS).result   # e.g. ['before_scan', 'after_scan']
dump_idx = context.get(ResultEnum.CALIBRATOR_DUMP_INDICES).result        # {period: dump_indices} in original obs numbering
noise_on_index = context.get(ResultEnum.NOISE_ON_INDEX).result           # firing positions in track-local index space

first_period = periods[0]
period_dumps = np.sort(np.asarray(dump_idx[first_period]))

# Convert each firing's track-local index to an original observation dump number.
# noise_on_index is a ratio-weighted float (a firing can straddle two dumps), so round first.
dumps = np.asarray(track_data._dumps())
firing_dump = dumps[np.round(noise_on_index).astype(int)]

# Split the period's dumps into contiguous pointing blocks, then tag each firing.
breaks = np.where(np.diff(period_dumps) > 1)[0]
pointing_blocks = np.split(period_dumps, breaks + 1)
pointing_id = np.full(len(firing_dump), -1, dtype=int)
for p, block in enumerate(pointing_blocks):
    pointing_id[np.isin(firing_dump, block)] = p
in_first = pointing_id >= 0

print(f"First period '{first_period}': {in_first.sum()} firings across {len(pointing_blocks)} pointings")

# Channel-median excess per firing, then percent deviation about the per-receiver
# mean over the first-period firings. Masked firings stay masked and are skipped by the plot.
median_excess = ma.median(nd_excess, axis=1)               # (n_firings, n_receivers)
mean_excess = ma.mean(median_excess[in_first], axis=0)     # per-receiver reference for the first period
pct_dev = 100.0 * (median_excess - mean_excess) / mean_excess
t_min = noise_on_timestamp / 60.0

# Mirror the median notebook's first plot: one dish, its two polarisations, points coloured by pointing.
dish = receiver_names[0][:-1]
dish_recv = [i for i, n in enumerate(receiver_names) if n.startswith(dish)]

cmap = plt.get_cmap('tab10')
fig, axes = plt.subplots(len(dish_recv), 1, sharex=True, figsize=(12, 3 * len(dish_recv)))
axes = np.atleast_1d(axes)
for ax, i_rec in zip(axes, dish_recv):
    for p in range(len(pointing_blocks)):
        sel = in_first & (pointing_id == p)
        if np.any(sel):
            ax.plot(t_min[sel], pct_dev[sel, i_rec], 'o-', color=cmap(p % 10),
                    ms=4, lw=0.8, label=f'Pointing {p}')
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_ylabel(f'{receiver_names[i_rec]}\ndeviation [%]')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_title(f"{dish} — % deviation of channel-median ND excess ({first_period})")
axes[-1].set_xlabel('Firing time [min since track start]')
plt.tight_layout()
plt.show()